In [21]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("data/education.db")

In [22]:
conn.executescript("""
DROP TABLE IF EXISTS primary_female_raw;
DROP TABLE IF EXISTS primary_male_raw;
DROP TABLE IF EXISTS primary_female_clean;
DROP TABLE IF EXISTS primary_male_clean;
DROP TABLE IF EXISTS primary_gender_merged;
DROP TABLE IF EXISTS primary_male_long_sql;
DROP TABLE IF EXISTS primary_female_long_sql;
""")

In [23]:
df_primary_female = pd.read_csv(
    "data/raw/enrollment/PRIMARY_Female_school_enrollment_DATA.csv",
    skiprows=4,
    engine="python"
)

df_primary_female.to_sql(
    "primary_female_raw",
    conn,
    if_exists="replace",
    index=False
)

pd.read_sql("SELECT COUNT(*) FROM primary_female_raw;", conn)

,COUNT(*)
0,266


In [24]:
df_primary_male = pd.read_csv(
    "data/raw/enrollment/PRIMARY_Male_school_enrollment_DATA.csv",
    skiprows=4,
    engine="python"
)

df_primary_male_clean = df_primary_male[
    ["Country Name", "Country Code", "Indicator Name",
     "2000","2001","2002","2003","2004","2005","2006","2007","2008","2009",
     "2010","2011","2012","2013","2014","2015","2016","2017","2018","2019",
     "2020","2021","2022","2023"]
]

df_primary_male_clean = df_primary_male_clean.dropna(how="all", subset=[
    "2000","2001","2002","2003","2004","2005","2006","2007","2008","2009",
    "2010","2011","2012","2013","2014","2015","2016","2017","2018","2019",
    "2020","2021","2022","2023"
])

df_primary_male_clean.to_sql(
    "primary_male_raw",
    conn,
    if_exists="replace",
    index=False
)

pd.read_sql("SELECT COUNT(*) FROM primary_male_raw;", conn)

,COUNT(*)
0,250


In [28]:
conn.executescript("""
DROP TABLE IF EXISTS primary_female_clean;
DROP TABLE IF EXISTS primary_male_clean;
""")

In [29]:
conn.executescript("""
CREATE TABLE primary_female_clean AS
SELECT
    "Country Name" AS country_name,
    "Country Code" AS country_code,
    "Indicator Name" AS indicator_name,

    "2000","2001","2002","2003","2004","2005","2006","2007","2008","2009",
    "2010","2011","2012","2013","2014","2015","2016","2017","2018","2019",
    "2020","2021","2022","2023"

FROM primary_female_raw

WHERE "Country Code" IN (
'AFE','AFW','ARB','AUS','EAS','EUU','LCN','NAC','SAS',
'LIC','LMC','UMC','HIC'
);
""")

pd.read_sql("SELECT COUNT(*) FROM primary_female_clean;", conn)

,COUNT(*)
0,13


In [30]:
conn.executescript("""
CREATE TABLE primary_male_clean AS
SELECT
    "Country Name" AS country_name,
    TRIM("Country Code") AS country_code,
    "Indicator Name" AS indicator_name,

    "2000","2001","2002","2003","2004","2005","2006","2007","2008","2009",
    "2010","2011","2012","2013","2014","2015","2016","2017","2018","2019",
    "2020","2021","2022","2023"

FROM primary_male_raw
WHERE TRIM("Country Code") IN (
'AFE','AFW','ARB','AUS','EAS','EUU','LCN','NAC','SAS',
'LIC','LMC','UMC','HIC'
);
""")

pd.read_sql("SELECT COUNT(*) FROM primary_male_clean;", conn)

,COUNT(*)
0,13


In [31]:
pd.read_sql("""
SELECT
  'male' AS table_name,
  COUNT(*) AS rows,
  MIN("2000") AS min_2000,
  MAX("2000") AS max_2000
FROM primary_male_clean

UNION ALL

SELECT
  'female',
  COUNT(*),
  MIN("2000"),
  MAX("2000")
FROM primary_female_clean;
""", conn)

,table_name,rows,min_2000,max_2000
0,male,13,77.034477,117.201439
1,female,13,60.934559,114.087532


In [32]:
conn.executescript("""
DROP TABLE IF EXISTS primary_male_long_sql;
DROP TABLE IF EXISTS primary_female_long_sql;

CREATE TABLE primary_male_long_sql AS
SELECT country_name, country_code, '2000' AS year, "2000" AS male_enrollment FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2001', "2001" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2002', "2002" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2003', "2003" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2004', "2004" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2005', "2005" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2006', "2006" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2007', "2007" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2008', "2008" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2009', "2009" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2010', "2010" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2011', "2011" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2012', "2012" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2013', "2013" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2014', "2014" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2015', "2015" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2016', "2016" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2017', "2017" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2018', "2018" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2019', "2019" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2020', "2020" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2021', "2021" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2022', "2022" FROM primary_male_clean
UNION ALL SELECT country_name, country_code, '2023', "2023" FROM primary_male_clean;

CREATE TABLE primary_female_long_sql AS
SELECT country_name, country_code, '2000' AS year, "2000" AS female_enrollment FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2001', "2001" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2002', "2002" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2003', "2003" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2004', "2004" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2005', "2005" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2006', "2006" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2007', "2007" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2008', "2008" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2009', "2009" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2010', "2010" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2011', "2011" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2012', "2012" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2013', "2013" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2014', "2014" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2015', "2015" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2016', "2016" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2017', "2017" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2018', "2018" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2019', "2019" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2020', "2020" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2021', "2021" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2022', "2022" FROM primary_female_clean
UNION ALL SELECT country_name, country_code, '2023', "2023" FROM primary_female_clean;
""")

In [33]:
conn.executescript("""
DROP TABLE IF EXISTS primary_gender_merged;

CREATE TABLE primary_gender_merged AS
SELECT
    m.country_name,
    m.country_code,
    m.year,
    m.male_enrollment,
    f.female_enrollment
FROM primary_male_long_sql m
JOIN primary_female_long_sql f
    ON m.country_code = f.country_code
    AND m.year = f.year;
""")

In [34]:
pd.read_sql("SELECT COUNT(*) FROM primary_gender_merged;", conn)

,COUNT(*)
0,312


In [35]:
pd.read_sql("SELECT * FROM primary_gender_merged LIMIT 10;", conn)

,country_name,country_code,year,male_enrollment,female_enrollment
0,Africa Eastern and Southern,AFE,2000,83.367851,74.106873
1,Africa Western and Central,AFW,2000,89.540253,71.428177
2,Arab World,ARB,2000,93.257462,82.658501
3,Australia,AUS,2000,101.746597,101.411880
4,East Asia & Pacific,EAS,2000,110.146797,110.308182
5,European Union,EUU,2000,103.935677,102.599060
6,High income,HIC,2000,103.613380,102.505058
7,Latin America & Caribbean,LCN,2000,117.201439,114.087532
8,Low income,LIC,2000,77.034477,60.934559
9,Lower middle income,LMC,2000,100.187172,87.068237


In [37]:
df = pd.read_sql("SELECT * FROM primary_gender_merged;", conn)
df.to_csv("data/cleaned/primary_gender_merged.csv", index=False)

pd.read_sql("SELECT * FROM primary_male_clean;", conn).to_csv(
    "data/cleaned/primary_male_clean.csv", index=False
)

pd.read_sql("SELECT * FROM primary_female_clean;", conn).to_csv(
    "data/cleaned/primary_female_clean.csv", index=False
)